In [2]:
from setup import *

## Metadaten für alle RIS laden

Entities sind die angebundenen RIS. Mit diesem Notebook downloaden wir alle entities, die in Poliscope aktuell vorlagen, inklusive Metadaten und ids.

Diese Liste ist nötig, um später gezielt für bestimmte Kommunen Daten zu finden, oder Rechercheergebnisse passend zu filtern.


In [3]:
# Alle Items sammeln
all_entities = []

# Paginierungsparameter
#achtung, tatsächlich ist gerade auf 300 Limit
limit = 500
offset = 0
total = 16079

print("/entities")

# Durch alle Pages iterieren
while offset < total:
    response = poliscope_request(
        "GET",
        "/entities",
        params={
            "limit": limit,
            "offset": offset,
            "detail": "standard",
        },
        timeout=10.0,
    )
    data = response.json()
    items = data.get("data", [])
    all_entities.extend(items)  # Alle Items zur Liste hinzufügen
    
    print(f"Downloaded {offset + len(items)} / {total} items")
    offset += limit

/entities
Downloaded 500 / 16079 items
Downloaded 1000 / 16079 items
Downloaded 1500 / 16079 items
Downloaded 2000 / 16079 items
Downloaded 2500 / 16079 items
Downloaded 3000 / 16079 items
Downloaded 3500 / 16079 items
Downloaded 4000 / 16079 items
Downloaded 4500 / 16079 items
Downloaded 5000 / 16079 items
Downloaded 5500 / 16079 items
Downloaded 6000 / 16079 items
Downloaded 6500 / 16079 items
Downloaded 7000 / 16079 items
Downloaded 7500 / 16079 items
Downloaded 8000 / 16079 items
Downloaded 8500 / 16079 items
Downloaded 9000 / 16079 items
Downloaded 9500 / 16079 items
Downloaded 10000 / 16079 items
Downloaded 10500 / 16079 items
Downloaded 11000 / 16079 items
Downloaded 11500 / 16079 items
Downloaded 12000 / 16079 items
Downloaded 12500 / 16079 items
Downloaded 13000 / 16079 items
Downloaded 13500 / 16079 items
Downloaded 14000 / 16079 items
Downloaded 14500 / 16079 items
Downloaded 15000 / 16079 items
Downloaded 15500 / 16079 items
Downloaded 16000 / 16079 items
Downloaded 16079 /

In [4]:
entities_df = pd.DataFrame(all_entities)
entities_df

,id,name,level,location,parents,population,area,postalCode,city,street,ris
0,072355007001,Aach,60,"{'lat': 49.789424896240234, 'lon': 6.590655803...","[{'id': '072355007', 'name': 'Trier-Land', 'le...",1142.0,6.96000,54295,Trier,Gartenfeldstraße 12,None
1,083355001001,"Aach, Stadt",60,"{'lat': 47.84281921386719, 'lon': 8.8508443832...","[{'id': '083355001', 'name': 'VVG der Stadt En...",2304.0,10.68000,78234,Engen,Hauptstraße 11,"{'id': 3206, 'status': 'active', 'urls': ['htt..."
2,053340002002,"Aachen, Stadt",60,"{'lat': 50.775428771972656, 'lon': 6.081491947...","[{'id': '053340002', 'name': 'Aachen, Stadt', ...",249070.0,160.85001,52058,Aachen,Markt,"{'id': 5373, 'status': 'active', 'urls': ['htt..."
3,053340002,"Aachen, Stadt",50,"{'lat': 50.7764646874706, 'lon': 6.08396351207...","[{'id': '05334', 'name': 'Städteregion Aachen'...",249070.0,NaN,52058,Aachen,Markt,None
4,081365001088,"Aalen, Stadt",60,"{'lat': 48.83597183227539, 'lon': 10.089909553...","[{'id': '081365001', 'name': 'VVG der Stadt Aa...",68351.0,146.58000,73430,Aalen,Marktplatz 30,"{'id': 1228, 'status': 'active', 'urls': ['htt..."
...,...,...,...,...,...,...,...,...,...,...,...
16074,082255006113,Zwingenberg,60,"{'lat': 49.41682434082031, 'lon': 9.0412893295...","[{'id': '082255006', 'name': 'GVV Neckargerach...",680.0,4.72000,69437,Neckargerach,Hauptstraße 25,None
16075,064310022022,"Zwingenberg, Stadt",60,"{'lat': 49.72316360473633, 'lon': 8.6126108169...","[{'id': '064310022', 'name': 'Zwingenberg, Sta...",7202.0,5.66000,64673,Zwingenberg,Untergasse 16,"{'id': 4627, 'status': 'active', 'urls': ['htt..."
16076,064310022,"Zwingenberg, Stadt",50,"{'lat': 49.723706, 'lon': 8.61302}","[{'id': '06431', 'name': 'Bergstraße', 'level'...",7202.0,NaN,64673,Zwingenberg,Untergasse 16,None
16077,145215140,Zwönitz,50,"{'lat': 50.630205, 'lon': 12.813659}","[{'id': '14521', 'name': 'Erzgebirgskreis', 'l...",14601.0,109.98000,08297,Zwönitz,Markt 6,None


In [5]:
entities_df["parents"][0]

[{'id': '072355007', 'name': 'Trier-Land', 'level': '50'},
 {'id': '07235', 'name': 'Trier-Saarburg', 'level': '40'},
 {'id': '07pr02', 'name': 'Trier', 'level': 'pr'},
 {'id': '07', 'name': 'Rheinland-Pfalz', 'level': '10'}]

In [6]:
entities_df.to_csv("./data/metadata/all_entities.csv", index=False)

# Entities filtern

Wir können die Entity-Liste zB mithilfe der Spalten "level", "population", "area", oder "id" (entspricht dem [ARS Schlüssel](https://www.destatis.de/DE/Themen/Laender-Regionen/Regionales/Gemeindeverzeichnis/Glossar/regionalschluessel.html)) filtern.




In [7]:
#füge Spalte hinzu, die für jede Entität angibt, ob sie in den Ost-Bundesländern liegt
ost_laender = ["Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen"]
def is_eastern(row):
    # Entität selbst ist ein Ost-Bundesland
    if row["name"] in ost_laender:
        return True

    # Einer der übergeordneten Einträge ist ein Ost-Bundesland
    parents = row["parents"]
    if isinstance(parents, list):
        return any(parent["name"] in ost_laender for parent in parents)

    return False

entities_df["is_eastern"] = entities_df.apply(is_eastern, axis=1)

Hinweis:

Die Poliscope-Entities sind ein vollständiges Abbild der deutschen Verwaltungseinheiten. Sprich: Jede Gemeinde, jeder Gemeindeverband, jede Planungsregion, jeder Landkreis und jedes Bunedsland sind mit Koordinaten, Namen ud ARS-Code vertreten. Aber nicht alle diese Ebenen haben eigene Kommunalparlamente, und nicht alle Kommunalparlamente pflegen ein digitales RIS, und nicht alle digitalen RIS sind für Poliscope auslesbar.

Trotzdem sind alle diese Entitäten im Datensatz enthalten, um zB vollständige Karten anzeigen zu können, oder Anteile korrekt berechnen zu können.

In [8]:
bundeslaender = entities_df[entities_df["level"] == "10"]
landkreise = entities_df[entities_df["level"] == "40"]
gemeindeverbaende = entities_df[entities_df["level"] == "50"]
gemeinden = entities_df[entities_df["level"] == "60"]
planungsregionen = entities_df[entities_df["level"] == "pr"]

print("Länder:", bundeslaender["ris"].isna().sum(), "von", len(bundeslaender), "ohne auslesbares RIS")
print("Landkreise:", landkreise["ris"].isna().sum(), "von", len(landkreise), "ohne auslesbares RIS")
print("Gemeindeverbände:", gemeindeverbaende["ris"].isna().sum(), "von", len(gemeindeverbaende), "ohne auslesbares RIS")
print("Gemeinden:", gemeinden["ris"].isna().sum(), "von", len(gemeinden), "ohne auslesbares RIS")
print("Planungsregionen:", planungsregionen["ris"].isna().sum(), "von", len(planungsregionen), "ohne auslesbares RIS")

Länder: 16 von 16 ohne auslesbares RIS
Landkreise: 1 von 400 ohne auslesbares RIS
Gemeindeverbände: 3734 von 4604 ohne auslesbares RIS
Gemeinden: 6902 von 10945 ohne auslesbares RIS
Planungsregionen: 7 von 113 ohne auslesbares RIS


## Exkurs: Grad der Verstädterung als Spalte ergänzen

In [9]:
#Quelle: https://www.destatis.de/DE/Themen/Laender-Regionen/Regionales/Gemeindeverzeichnis/_inhalt.html#101366

gv = pd.read_excel("./data/metadata/gemeindeverzeichnis_2026-05-26.xlsx", sheet_name="Onlineprodukt_Gemeinden30062026",dtype="str", skiprows=2).dropna(how="all")

In [10]:
#ars ist in mehrere Spalten aufgeteilt - zusammenführen
gv["gvs"] = (
    gv["Amtlicher Regionalschlüssel (ARS)"].fillna("").astype(str)
    + gv["Unnamed: 3"].fillna("").astype(str)
    + gv["Unnamed: 4"].fillna("").astype(str)
    + gv["Unnamed: 5"].fillna("").astype(str)
)

In [11]:
#gängige label für Grad der Verstädterung als Spalte ergänzen
gv["degurba"] = gv["Unnamed: 19"].apply(lambda x: "urban" if x == "dicht besiedelt" else
                                        ("suburban" if x == "mittlere Besiedlungsdichte" else
                                         ("rural" if x == "gering besiedelt" else None)))

In [12]:
#in den metadaten ablegen
gv[["gvs", "degurba"]].to_csv("./data/metadata/ars-degurba.csv", index=False)

In [13]:
gv["gvs"] = gv["gvs"].astype("string")
entities_df["id"] = entities_df["id"].astype("string")

# Match über die ersten 9 Stellen
gv["match"] = gv["gvs"].str[:9]
entities_df["match"] = entities_df["id"].str[:9]

# Nur Zeilen mit vorhandenem degurba verwenden
degurba_map = (
    gv[gv["degurba"].notna()]
    .drop_duplicates("match")
    .set_index("match")["degurba"]
)

# degurba übernehmen
entities_df["degurba"] = entities_df["match"].map(degurba_map)

# Hilfsspalte entfernen
entities_df.drop(columns="match", inplace=True)
gv.drop(columns="match", inplace=True)

In [14]:
entities_df.to_csv("./data/metadata/all_entities.csv", index=False)